
# Disneyland Customer Experience Analysis

This notebook focuses on the original assignment deliverable: cleaning the Disneyland review data, exploring metadata patterns, extracting practical customer-experience insights, and summarizing the final QA system at a high level.



## What This Notebook Covers

- Clean and standardize the Disneyland review dataset
- Explore branch, timing, and reviewer-location patterns
- Use lightweight text signals to identify operational pain points
- Pair each important finding with a concrete recommendation
- Summarize the final QA system and evaluation results

**Core design principle:** Code calculates the facts, embeddings retrieve the evidence, and the LLM explains the result.


In [ ]:

from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

from src.config import get_settings
from src.data import load_clean_reviews, load_reviews_csv

pd.options.display.max_columns = 50
pd.options.display.max_colwidth = 120
plt.style.use('seaborn-v0_8-whitegrid')

DATASET_PATH = Path('DisneylandReviews.csv')
raw_df, detected_encoding = load_reviews_csv(DATASET_PATH)
reviews_df = load_clean_reviews(DATASET_PATH)
settings = get_settings()

BRANCH_ORDER = [
    'Disneyland_California',
    'Disneyland_HongKong',
    'Disneyland_Paris',
]
MISSING_TOKENS = {'', 'missing', 'nan', 'none', 'null', 'na', 'n/a'}


## Data Preparation And Quality Checks

In [ ]:

raw_year_month = raw_df['Year_Month'].astype('string').str.strip().str.lower()
raw_year_month = raw_year_month.mask(raw_year_month.isin(MISSING_TOKENS))

quality_summary = pd.DataFrame(
    [
        ('Raw rows', len(raw_df)),
        ('Cleaned rows', len(reviews_df)),
        ('Exact duplicate rows removed', int(raw_df.duplicated().sum())),
        ('Duplicate Review_ID rows removed', int(raw_df['Review_ID'].duplicated().sum())),
        ('Rows with parsed review dates', int(reviews_df['Review_Date'].notna().sum())),
        ('Rows with missing review dates', int(reviews_df['Review_Date'].isna().sum())),
        ('Reviewer locations standardized to Unknown', int((reviews_df['Reviewer_Location'] == 'Unknown').sum())),
        ('Detected CSV encoding', detected_encoding),
    ],
    columns=['Metric', 'Value'],
)

display(quality_summary)
display(Markdown(
    f"The raw file loaded successfully with **{detected_encoding}**. After removing exact duplicates and duplicate `Review_ID` rows, the working dataset contains **{len(reviews_df):,} cleaned reviews**."
))


In [ ]:

column_snapshot = pd.DataFrame({
    'dtype': reviews_df.dtypes.astype(str),
    'missing_values': reviews_df.isna().sum(),
})

profile_snapshot = pd.DataFrame(
    {
        'Branches': [', '.join(sorted(reviews_df['Branch'].dropna().unique()))],
        'Rating values': [', '.join(map(str, sorted(reviews_df['Rating'].dropna().unique())))],
        'Date coverage': [
            f"{reviews_df['Review_Date'].dropna().min():%Y-%m} to {reviews_df['Review_Date'].dropna().max():%Y-%m}"
        ],
        'Top reviewer locations': [
            ', '.join(reviews_df['Reviewer_Location'].value_counts().head(5).index.tolist())
        ],
    }
)

display(column_snapshot.loc[['Review_ID', 'Rating', 'Year_Month', 'Reviewer_Location', 'Review_Text', 'Branch', 'Review_Date', 'Season', 'Rating_Sentiment']])
display(profile_snapshot)
display(reviews_df[['Review_ID', 'Rating', 'Year_Month', 'Reviewer_Location', 'Branch', 'Review_Text']].head(3))



### Cleaning Decisions

- Column names and string fields were stripped for whitespace consistency.
- Exact duplicate rows were removed.
- Duplicate `Review_ID` records were de-duplicated conservatively by keeping the fuller non-truncated review when a shortened `...More` version also appeared.
- `Rating` was converted to numeric and `Year_Month` was parsed into `Review_Date`, `Year`, `Month`, `Month_Name`, and `Season`.
- Missing reviewer locations were standardized to `Unknown`.
- Review text was cleaned only lightly to preserve punctuation and meaning.
- `Rating_Sentiment` maps 1-2 to negative, 3 to neutral, and 4-5 to positive.


## Metadata Exploration

In [ ]:

branch_summary = (
    reviews_df.groupby('Branch')
    .agg(
        Review_Count=('Review_ID', 'count'),
        Average_Rating=('Rating', 'mean'),
        Median_Rating=('Rating', 'median'),
        Positive_Share=('Rating_Sentiment', lambda s: (s == 'positive').mean() * 100),
        Neutral_Share=('Rating_Sentiment', lambda s: (s == 'neutral').mean() * 100),
        Negative_Share=('Rating_Sentiment', lambda s: (s == 'negative').mean() * 100),
    )
    .reindex(BRANCH_ORDER)
    .round({'Average_Rating': 3, 'Median_Rating': 1, 'Positive_Share': 1, 'Neutral_Share': 1, 'Negative_Share': 1})
)

display(branch_summary)
display(Markdown(
    'California has the strongest headline rating profile, while Paris has the weakest average and the heaviest negative tail. Hong Kong sits between them on most branch-level measures.'
))


In [ ]:

fig, ax = plt.subplots(figsize=(8, 4.5))
volume_by_branch = branch_summary['Review_Count']
colors = ['#355070', '#6d597a', '#b56576']
volume_by_branch.plot(kind='bar', ax=ax, color=colors)
ax.set_title('Which Parks Dominate The Review Dataset?')
ax.set_xlabel('Branch')
ax.set_ylabel('Review count')
ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha='right')
for patch in ax.patches:
    ax.annotate(f"{int(patch.get_height()):,}", (patch.get_x() + patch.get_width() / 2, patch.get_height()),
                xytext=(0, 5), textcoords='offset points', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

display(Markdown(
    'California contributes the largest share of the evidence base, followed by Paris and then Hong Kong. That matters because smaller samples can make a branch look more volatile than it really is.'
))


In [ ]:

rating_profile = (
    reviews_df.assign(Rating=reviews_df['Rating'].astype(int))
    .pivot_table(index='Branch', columns='Rating', values='Review_ID', aggfunc='count', fill_value=0)
    .reindex(BRANCH_ORDER)
)
rating_profile_pct = rating_profile.div(rating_profile.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(8.5, 4.8))
rating_profile_pct.plot(kind='bar', stacked=True, ax=ax, cmap='RdYlGn')
ax.set_title("How Does Each Park's 1-5 Rating Mix Differ?")
ax.set_xlabel('Branch')
ax.set_ylabel('Share of reviews (%)')
ax.legend(title='Rating', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha='right')
plt.tight_layout()
plt.show()

display(Markdown(
    'The average alone hides shape differences. California has the strongest 5-star concentration, while Paris carries the largest 1-2 star share and therefore the heaviest downside risk.'
))


In [ ]:

sentiment_profile = branch_summary[['Negative_Share', 'Neutral_Share', 'Positive_Share']]

fig, ax = plt.subplots(figsize=(8.5, 4.5))
sentiment_profile.plot(kind='bar', stacked=True, ax=ax, color=['#c1121f', '#fcbf49', '#588157'])
ax.set_title('Where Are Positive, Neutral, And Negative Reviews Concentrated?')
ax.set_xlabel('Branch')
ax.set_ylabel('Share of reviews (%)')
ax.legend(title='Sentiment', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha='right')
plt.tight_layout()
plt.show()

display(Markdown(
    'Paris has the lowest positive share and the largest negative share. Hong Kong is more positive than Paris, but it still trails California on overall sentiment quality.'
))


In [ ]:

dated_reviews = reviews_df.dropna(subset=['Review_Date']).copy()
monthly_trends = (
    dated_reviews.groupby(['Branch', pd.Grouper(key='Review_Date', freq='ME')])
    .agg(Average_Rating=('Rating', 'mean'), Review_Count=('Review_ID', 'count'))
    .reset_index()
)
monthly_trends['Rolling_3M_Rating'] = (
    monthly_trends.sort_values('Review_Date')
    .groupby('Branch')['Average_Rating']
    .transform(lambda s: s.rolling(3, min_periods=1).mean())
)

fig, ax = plt.subplots(figsize=(9.5, 4.8))
for branch, color in zip(BRANCH_ORDER, ['#355070', '#6d597a', '#b56576']):
    branch_df = monthly_trends[monthly_trends['Branch'] == branch]
    ax.plot(branch_df['Review_Date'], branch_df['Rolling_3M_Rating'], label=branch, linewidth=2, color=color)
ax.set_title('How Has Average Rating Moved Over Time? (3-Month Rolling Average)')
ax.set_xlabel('Review month')
ax.set_ylabel('Average rating')
ax.legend()
plt.tight_layout()
plt.show()

missing_dates = int(reviews_df['Review_Date'].isna().sum())
display(Markdown(
    f'Only rows with valid dates are used in the time chart, so **{missing_dates:,} reviews** are excluded from time-based analysis. The branch-level curves shift over time, but the notebook does not assume any single operational cause.'
))


In [ ]:

season_stats = (
    dated_reviews.groupby(['Branch', 'Season'])
    .agg(Review_Count=('Review_ID', 'count'), Average_Rating=('Rating', 'mean'))
    .reset_index()
)
season_order = ['Winter', 'Spring', 'Summer', 'Autumn']
season_stats = season_stats[season_stats['Review_Count'] >= 200].copy()
season_stats['Season'] = pd.Categorical(season_stats['Season'], categories=season_order, ordered=True)
season_stats = season_stats.sort_values(['Season', 'Branch'])

fig, ax = plt.subplots(figsize=(8.5, 4.5))
for branch, color in zip(BRANCH_ORDER, ['#355070', '#6d597a', '#b56576']):
    branch_df = season_stats[season_stats['Branch'] == branch]
    ax.plot(branch_df['Season'], branch_df['Average_Rating'], marker='o', linewidth=2, label=branch, color=color)
ax.set_title('Which Seasons Look Stronger Or Weaker By Park?')
ax.set_xlabel('Season')
ax.set_ylabel('Average rating')
ax.legend()
plt.tight_layout()
plt.show()

display(season_stats.pivot(index='Season', columns='Branch', values=['Review_Count', 'Average_Rating']).round(3))
display(Markdown(
    'These seasons are simple calendar buckets, so they are useful for pattern-finding but they do not fully reflect local climate or holiday calendars.'
))


In [ ]:

top_locations = (
    reviews_df.groupby('Reviewer_Location')
    .agg(Review_Count=('Review_ID', 'count'), Average_Rating=('Rating', 'mean'))
    .query('Review_Count >= 30')
    .sort_values('Review_Count', ascending=False)
    .head(10)
    .round({'Average_Rating': 3})
)

display(top_locations)
display(Markdown(
    'The United States and the United Kingdom dominate reviewer volume, with Australia and Canada forming the next-largest English-language segments. Those segments are especially useful for later QA demonstrations because they have both scale and distinct viewpoints.'
))


## Text-Based Customer Experience Insights

In [ ]:

ASPECT_PATTERNS = {
    'Crowding and queues': r'\b(queue|queues|queued|wait|waiting|line|lines|crowd|crowded|busy)\b',
    'Staff and service': r'\b(staff|service|employee|employees|cast member|friendly|helpful|rude)\b',
    'Food and dining': r'\b(food|meal|restaurant|dining|snack|burger|pizza|drink|breakfast|lunch|dinner)\b',
    'Price and value': r'\b(price|priced|expensive|cheap|value|worth|ticket|tickets|cost)\b',
    'Rides and attractions': r'\b(ride|rides|attraction|attractions|coaster|roller coaster|show|shows)\b',
    'Cleanliness and maintenance': r'\b(clean|dirty|maintenance|closed|closure|broken|repair|renovation)\b',
    'Family experience': r'\b(family|kids|children|child|baby|stroller)\b',
}

aspect_flags = pd.DataFrame(index=reviews_df.index)
for aspect, pattern in ASPECT_PATTERNS.items():
    aspect_flags[aspect] = reviews_df['Review_Text'].str.contains(pattern, case=False, regex=True, na=False)

branch_negative_baseline = reviews_df.groupby('Branch')['Rating_Sentiment'].apply(lambda s: (s == 'negative').mean() * 100)
records = []
for branch in BRANCH_ORDER:
    branch_mask = reviews_df['Branch'] == branch
    branch_total = int(branch_mask.sum())
    for aspect in ASPECT_PATTERNS:
        mentioned = branch_mask & aspect_flags[aspect]
        subset = reviews_df.loc[mentioned]
        if subset.empty:
            continue
        records.append(
            {
                'Branch': branch,
                'Aspect': aspect,
                'Review_Count': int(len(subset)),
                'Mention_Rate': len(subset) / branch_total * 100,
                'Average_Rating': subset['Rating'].mean(),
                'Negative_Share': (subset['Rating_Sentiment'] == 'negative').mean() * 100,
                'Negative_Share_Gap_vs_Branch': ((subset['Rating_Sentiment'] == 'negative').mean() * 100) - branch_negative_baseline[branch],
            }
        )

branch_aspect_summary = pd.DataFrame(records).sort_values(['Branch', 'Negative_Share_Gap_vs_Branch', 'Review_Count'], ascending=[True, False, False])
priority_issues = (
    branch_aspect_summary.query('Review_Count >= 200')
    .groupby('Branch', group_keys=False)
    .head(2)
    .copy()
)

display(priority_issues[['Branch', 'Aspect', 'Review_Count', 'Mention_Rate', 'Average_Rating', 'Negative_Share', 'Negative_Share_Gap_vs_Branch']].round(2))
display(Markdown(
    'This section uses transparent keyword-based aspect tagging rather than a black-box topic model. The goal is not perfect labeling, but a fast, explainable read on where dissatisfaction clusters inside each park.'
))


In [ ]:

RECOMMENDATION_TEMPLATES = {
    'Crowding and queues': 'Tighten queue communication, peak-time staffing, and virtual or timed-entry guidance in the most complaint-heavy windows.',
    'Staff and service': 'Prioritize frontline coaching and service-recovery scripts where staff mentions carry an unusually high negative share.',
    'Food and dining': 'Review menu value, dining throughput, and expectation setting where food is mentioned often but rated poorly.',
    'Price and value': 'Clarify what is included in the experience and test value-oriented bundles or messaging for cost-sensitive visitors.',
    'Rides and attractions': 'Focus on attraction uptime and clearer closure communication where ride mentions drive frustration.',
    'Cleanliness and maintenance': 'Increase visible housekeeping and preventative maintenance in the branches with elevated cleanliness or closure complaints.',
    'Family experience': 'Improve family routing, stroller convenience, and child-focused wayfinding where family mentions are common but satisfaction drops.',
}

priority_issues = priority_issues.copy()
priority_issues['Recommendation'] = priority_issues['Aspect'].map(RECOMMENDATION_TEMPLATES)

display(priority_issues[['Branch', 'Aspect', 'Review_Count', 'Negative_Share', 'Recommendation']].round(2))

recommendation_lines = []
for row in priority_issues.itertuples(index=False):
    recommendation_lines.append(
        f"- **{row.Branch} — {row.Aspect}:** {row.Recommendation} ({row.Review_Count:,} tagged reviews; negative share {row.Negative_Share:.1f}%)."
    )

display(Markdown('## Actionable Recommendations\n' + '\n'.join(recommendation_lines)))



## Final QA System Design

![Adaptive analytical RAG architecture](docs/architecture.svg)

**Final workflow**

Question  
→ LLM structured planner  
→ LangGraph orchestration  
→ pandas deterministic analytics  
→ Chroma semantic retrieval  
→ grounded LLM answer

**Key principle:** Code calculates the facts, embeddings retrieve the evidence, and the LLM explains the result.


In [ ]:

manifest_path = Path('artifacts/chroma/disney_reviews/manifest.json')
manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}

system_summary = pd.DataFrame(
    [
        ('Planner model', settings.planner_model),
        ('Answer model', settings.answer_model),
        ('Judge model', settings.judge_model),
        ('Embedding provider', settings.embedding_provider),
        ('Embedding model', manifest.get('local_embedding_model') or manifest.get('openai_embedding_model', 'Unavailable')),
        ('Indexed reviews', manifest.get('row_count', 'Unavailable')),
    ],
    columns=['Component', 'Configuration'],
)

display(system_summary)



## Evaluation Snapshot

- Deterministic benchmark: **10 / 10 cases passed**
- LLM judge sample: **5 representative questions**
- Mean judge scores: relevance **5.0**, faithfulness **4.6**, completeness **4.8**, clarity **5.0**, limitation handling **5.0**
- External-knowledge leaks detected: **0**

The judge is useful for checking whether an answer stays grounded in the supplied metrics and excerpts, but it is not treated as unquestionable ground truth.



## Limitations

- The review dataset is historical, so it should not be treated as a live operations dashboard.
- Seasonal categories are calendar-based and do not fully represent local climate or holiday effects.
- Keyword-based aspect tagging is intentionally simple and explainable, but it will miss some nuance and synonyms.
- The QA system is designed for grounded analysis, not for answering questions that require ticket prices, live crowd levels, weather forecasts, or other external data.
